# 02. EDA y Preparación de Datos

Este módulo se encarga del control de calidad de los datos cargados y de la generación de los ficheros de precios necesarios para el backtesting y el cálculo de señales de momentum.

## Objetivos del Notebook
1.  **Auditoría de Integridad**: Análisis ticker por ticker para detectar huecos dentro del periodo de vida del activo, precios negativos o saltos extremos (>50%).
2.  **Prevención del Sesgo de Supervivencia**: Validación de tener precios estrictamente dentro del rango operativo real de cada ticker, evitando "inventar" datos cuando una empresa se encuentra temporalmente/definitivamente excluida del índice.
3.  **Preparación de Ficheros Finales**:
    *   `momentum_execution_prices.csv`: Precios diarios nominales (Open/Close) para el motor de ejecución.
    *   `momentum_monthly_prices.csv`: Precios ajustados mensuales para el cálculo de señales.

## Flujo de Calidad
*   **Huecos Estructurales**: Se verifica que no falten datos técnicos si existe cotización ajustada.
*   **Consistencia Horizontal**: Validación de que para cada fecha existan Open, Close y Adj Close simultáneamente.
*   **Saltos de Datos**: Identificación de posibles errores en splits o ajustes de dividendos mediante el análisis de retornos extremos superiores al 50%.

## Preparación para la Estrategia
*   **Resampling Mensual**: Se utiliza la frecuencia Mensual para capturar el último precio ajustado de cada mes, base del cálculo de los periodos de 6 y 12 meses de Momentum con el lag requerido.
*   **Aplanado de Ejecución**: Se genera un archivo diario con sufijos `_OPEN` y `_CLOSE` para que el Notebook 04 pueda simular compras y ventas de forma precisa.

## Salidas Generadas
*   `Datos/momentum_execution_prices.csv`
*   `Datos/momentum_monthly_prices.csv`


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Configuración de visualización
plt.style.use('dark_background')
sns.set_palette("viridis")

# 1. Cargar datos con multi-índice (Atributo, Ticker)
raw_data = pd.read_csv('Datos/momentum_raw_data.csv', header=[0, 1], index_col=0, parse_dates=True)

adj_close = raw_data['Adj Close']
close_nom = raw_data['Close']
open_nom = raw_data['Open']
inSP500 = raw_data['In SP500']

print(f"📌 Datos cargados: {adj_close.shape[0]} días para {adj_close.shape[1]} activos.")

📌 Datos cargados: 9087 días para 1278 activos.


## 1. Análisis de Calidad y Limpieza

### Auditoría de Integridad (Per-Ticker)
Realizamos un análisis exhaustivo ticker por ticker. Esto es crucial para evitar que los nulos estructurales (antes de que una empresa entre en el índice o después de que salga) falseen las métricas de calidad.

In [ ]:
def perform_full_audit(adj, close, ope, inSP500):
    audit_results = []
    jump_details = []

    for ticker in adj.columns:
        # Identificar ventana operativa real del activo
        start_idx = adj[ticker].first_valid_index()
        end_idx = adj[ticker].last_valid_index()

        if start_idx is None: continue

        # Slicing de la vida del activo
        s_adj = adj[ticker].loc[start_idx:end_idx]
        s_close = close[ticker].loc[start_idx:end_idx]
        s_open = ope[ticker].loc[start_idx:end_idx]
        s_inSP500 = inSP500[ticker].loc[start_idx:end_idx]
        s_inSP500.fillna(3, inplace=True)


        # 2. Inconsistencia entre atributos (Si falta uno pero está el otro)
        consistency_errors = (s_adj.isna() != s_close.isna()).sum() + (s_adj.isna() != s_open.isna()).sum()

        # 3. Precios no positivos
        invalid_p = (s_adj <= 0).sum() + (s_close <= 0).sum() + (s_open <= 0).sum()

        # 4. Sin precio mientras la empresa está en el SP500
        inout_positive = np.logical_and(s_close.isna(), s_inSP500==1).sum()

        # 5. Saltos Diarios (>50%)
        # Solo calculamos retornos sobre datos válidos de este ticker
        rets = s_adj.pct_change().dropna()
        jumps = rets[rets.abs() > 1]

        if len(jumps) > 0:
            for date, val in jumps.items():
                jump_details.append({'ticker': ticker, 'date': date, 'return': val})


        if consistency_errors > 0 or invalid_p > 0 or len(jumps) > 0:
            audit_results.append({
                'ticker': ticker,
                'errores_consistencia': consistency_errors,
                'precios_invalidos': invalid_p,
                'diasEnIndice_sin_precio': inout_positive,
                'saltos_extremos': len(jumps),
            })

    return pd.DataFrame(audit_results), pd.DataFrame(jump_details)

df_audit, df_jumps = perform_full_audit(adj_close, close_nom, open_nom, inSP500)

if df_audit.empty:
    print("✅ Auditoría Completa: Todas las series son íntegras y consistentes.")
else:
    print(f"⚠️ Se han detectado anomalías en {len(df_audit)} activos.")
    display(df_audit)

if not df_jumps.empty:
    print(f"\n⚡ Total de saltos diarios > 50%: {len(df_jumps)}")
    # Inspeccionar el primer salto real encontrado
    ex = df_jumps.iloc[0]
    print(f"🔍 Ejemplo de inspección: {ex['ticker']} el {ex['date'].date()} (Retorno: {ex['return']:.2%})")
    display(raw_data.xs(ex['ticker'], axis=1, level=1).loc[ex['date'] - pd.Timedelta(days=5) : ex['date'] + pd.Timedelta(days=5)])

⚠️ Se han detectado anomalías en 1 activos.


,ticker,errores_consistencia,precios_invalidos,diasEnIndice_sin_precio,saltos_extremos
0,HIG,0,0,0,1



⚡ Total de saltos diarios > 50%: 1
🔍 Ejemplo de inspección: HIG el 2008-12-05 (Retorno: 102.36%)


,Adj Close,Close,Open,In SP500
date,,,,
2008-12-01,4.753635,6.61,7.880000,1.0
2008-12-02,4.854317,6.75,6.950000,1.0
2008-12-03,4.976574,6.92,6.429999,1.0
2008-12-04,5.185130,7.21,6.769999,1.0
2008-12-05,10.492517,14.59,9.299999,1.0
2008-12-08,10.686688,14.86,16.890000,1.0
2008-12-09,10.895245,15.15,14.689999,1.0
2008-12-10,11.226057,15.61,15.660000,1.0


## 2. Preparación de Ficheros Finales

In [ ]:
# 2. Generar Precios de Ejecución (Aplanando MultiIndex)
# Formato: TICKER_OPEN, TICKER_CLOSE. Ordenamos por ticker.
execution_prices = pd.concat([
    open_nom.add_suffix('_OPEN'), 
    close_nom.add_suffix('_CLOSE')
], axis=1)

# Ordenamos columnas para que cada activo tenga su Open/Close contiguo
execution_prices = execution_prices.reindex(sorted(execution_prices.columns), axis=1)
execution_prices.to_csv('Datos/momentum_execution_prices.csv')

# 3. Precios mensuales
# 1. Agrupamos por mes y año, y extraemos el último índice (fecha) de cada grupo
last_days_indices = adj_close.groupby(adj_close.index.to_period('M')).apply(lambda x: x.index[-1])
# 2. Filtramos el DataFrame original usando esos índices exactos
adj_close_monthly = adj_close.loc[last_days_indices]
adj_close_monthly.to_csv('Datos/momentum_monthly_prices.csv')

print("✅ Archivos generados correctamente en 'Datos/'.")
print(f"📊 Series procesadas: {adj_close_monthly.shape[1]} activos.")

✅ Archivos generados correctamente en 'Datos/'.
📊 Series procesadas: 1278 activos.


In [ ]:
# 1. Listar los niveles superiores del MultiÍndice (Atributos)
print("📌 Atributos disponibles:", raw_data.columns.get_level_values(0).unique().tolist())

# 2. Ticker de ejemplo para la prueba
ticker_test = 'AGN-201503'

    

adj_close_monthly.loc['2015-02-20':'2015-05-31', ticker_test]

📌 Atributos disponibles: ['Adj Close', 'Close', 'Open', 'In SP500']


date
2015-02-27    232.74
2015-03-31       NaN
2015-04-30       NaN
2015-05-29       NaN
Name: AGN-201503, dtype: float64